# Blog 12 — Auto Loader & Structured Streaming

## Practical, Core Data Engineer Notebook

### What this notebook is designed to teach

This notebook focuses on the **core Auto Loader + Structured Streaming workflow** without advanced streaming experiments.

You will actually:

1. Create a governed Unity Catalog Volume location
2. Create sample JSON source files
3. Read files with Auto Loader
4. Use a schema location
5. Write incrementally to a Delta Bronze table
6. Use a checkpoint
7. Demonstrate `availableNow`
8. Add another source file
9. Re-run using the **same checkpoint**
10. Observe incremental ingestion
11. Inspect the resulting Delta table
12. Understand schema hints
13. Understand schema evolution conceptually
14. Understand event time
15. Understand watermarks conceptually
16. Connect Auto Loader to Jobs and Unity Catalog

### Important

This notebook uses **real names from your Databricks environment** rather than placeholder paths such as:

```text
/Volumes/<catalog>/<schema>/<volume>
```

The notebook assumes the following Unity Catalog objects, which you have already been using:

```text
Catalog : workspace
Schema  : blog10_autoloader_streaming
Volume  : blog10_volume
```

If those objects do not exist, the setup cell will stop with a clear message.

# 1. The architecture

The workflow we will build is:

```text
Unity Catalog Volume
        │
        │ JSON files
        ▼
   Auto Loader
        │
        ▼
Structured Streaming
        │
        ▼
   Delta Bronze
```

Supporting state:

```text
Schema Location
      ↓
source schema information

Checkpoint
      ↓
streaming progress/state
```

Later, the broader architecture becomes:

```text
Cloud Files
    ↓
Auto Loader
    ↓
Bronze
    ↓
Silver
    ↓
Gold
    ↓
Jobs
```


# 2. Batch vs incremental processing

### Batch

```text
Read all available files
        ↓
Process
        ↓
Finish
```

### Incremental

```text
New files arrive
       ↓
Discover new files
       ↓
Process new data
       ↓
Remember progress
       ↓
More files arrive
       ↓
Process new data
```

Auto Loader is designed for the second pattern.

The important idea is:

> **Do not repeatedly treat the entire cloud-file directory as brand-new data.**

# 3. Real Unity Catalog configuration

We will use the same Unity Catalog structure you have already been working with.

```text
workspace
└── blog10_autoloader_streaming
    └── blog10_volume
```

We will create separate directories inside the Volume:

```text
source/
schemas/
checkpoints/
```

This keeps the demonstration isolated and avoids the previous problem of mixing old runs with new runs.

In [0]:
from pyspark.sql import functions as F
import json
from datetime import datetime

# ---------------------------------------------------------
# REAL UC NAMES USED IN YOUR WORKSPACE
# ---------------------------------------------------------

CATALOG = "workspace"
SCHEMA = "blog10_autoloader_streaming"
VOLUME = "blog10_volume"

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# One isolated run directory.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_ROOT = f"{VOLUME_ROOT}/blog12_runs/{RUN_ID}"

SOURCE_PATH = f"{RUN_ROOT}/source"
SCHEMA_PATH = f"{RUN_ROOT}/schema"
CHECKPOINT_PATH = f"{RUN_ROOT}/checkpoint"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.blog12_orders_bronze_{RUN_ID}"

print("Catalog       :", CATALOG)
print("Schema        :", SCHEMA)
print("Volume        :", VOLUME)
print("Run ID        :", RUN_ID)
print("Source        :", SOURCE_PATH)
print("Schema        :", SCHEMA_PATH)
print("Checkpoint    :", CHECKPOINT_PATH)
print("Bronze table  :", BRONZE_TABLE)


Catalog       : workspace
Schema        : blog10_autoloader_streaming
Volume        : blog10_volume
Run ID        : 20260824_111220
Source        : /Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source
Schema        : /Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/schema
Checkpoint    : /Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/checkpoint
Bronze table  : workspace.blog10_autoloader_streaming.blog12_orders_bronze_20260824_111220


# 4. Verify the Unity Catalog Volume

Before doing anything else, verify that the Volume exists.

This is important because Unity Catalog paths must follow:

```text
/Volumes/<catalog>/<schema>/<volume>/
```

We are **not** using DBFS root and we are not using placeholder namespace values.

In [0]:
# Verify the Volume exists.

try:
    volume_entries = dbutils.fs.ls(VOLUME_ROOT)
    print("PASS — Unity Catalog Volume is accessible.")
    print("Volume:", VOLUME_ROOT)
except Exception as e:
    raise RuntimeError(
        f"Could not access the Unity Catalog Volume: {VOLUME_ROOT}\n"
        "Verify that the catalog, schema and volume names are correct."
    ) from e


PASS — Unity Catalog Volume is accessible.
Volume: /Volumes/workspace/blog10_autoloader_streaming/blog10_volume


# 5. Create isolated run directories

Each execution gets its own source, schema and checkpoint locations.

This is intentionally different from using one permanent demo directory.

Why?

Because reusing old checkpoints can make a teaching notebook appear to ingest the wrong number of rows.

```text
Run A
 ├── source
 ├── schema
 └── checkpoint

Run B
 ├── source
 ├── schema
 └── checkpoint
```

Each run starts clean.

In [0]:
dbutils.fs.mkdirs(SOURCE_PATH)
dbutils.fs.mkdirs(SCHEMA_PATH)
dbutils.fs.mkdirs(CHECKPOINT_PATH)

print("Created isolated run directories.")


Created isolated run directories.


# 6. Create the initial source files

We will create exactly five orders in three JSON files.

```text
orders_001.json → 2 rows
orders_002.json → 2 rows
orders_003.json → 1 row

Total → 5 rows
```

These files simulate data arriving in cloud storage.

In [0]:
orders = {
    "orders_001.json": [
        {
            "order_id": 1001,
            "customer_id": 501,
            "amount": 1250.50,
            "order_status": "COMPLETED",
            "event_time": "2026-08-24T09:00:00"
        },
        {
            "order_id": 1002,
            "customer_id": 502,
            "amount": 850.00,
            "order_status": "PENDING",
            "event_time": "2026-08-24T09:02:00"
        }
    ],
    "orders_002.json": [
        {
            "order_id": 1003,
            "customer_id": 503,
            "amount": 420.75,
            "order_status": "COMPLETED",
            "event_time": "2026-08-24T09:05:00"
        },
        {
            "order_id": 1004,
            "customer_id": 504,
            "amount": 990.00,
            "order_status": "CANCELLED",
            "event_time": "2026-08-24T09:07:00"
        }
    ],
    "orders_003.json": [
        {
            "order_id": 1005,
            "customer_id": 505,
            "amount": 1750.25,
            "order_status": "COMPLETED",
            "event_time": "2026-08-24T09:10:00"
        }
    ]
}

for filename, records in orders.items():
    dbutils.fs.put(
        f"{SOURCE_PATH}/{filename}",
        "\n".join(json.dumps(record) for record in records),
        True
    )

files = dbutils.fs.ls(SOURCE_PATH)

print("Source files:", len(files))

for file_info in files:
    print(file_info.name, "|", file_info.size, "bytes")

assert len(files) == 3

print("PASS — exactly 3 source files were created.")

Wrote 242 bytes.
Wrote 244 bytes.
Wrote 123 bytes.
Source files: 3
orders_001.json | 242 bytes
orders_002.json | 244 bytes
orders_003.json | 123 bytes
PASS — exactly 3 source files were created.


# 7. Validate the source before Auto Loader

Before introducing streaming, check the raw source directly.

This is useful because it separates:

```text
Source problem
```

from:

```text
Streaming problem
```

We expect:

```text
Rows          = 5
Distinct IDs  = 5


In [0]:
direct_df = (
    spark.read
        .format("json")
        .load(SOURCE_PATH)
)

direct_rows = direct_df.count()
direct_ids = direct_df.select("order_id").distinct().count()

print("Direct source rows :", direct_rows)
print("Distinct order IDs :", direct_ids)

assert direct_rows == 5
assert direct_ids == 5

print("PASS — raw source contains exactly 5 unique orders.")
display(direct_df.orderBy("order_id"))


Direct source rows : 5
Distinct order IDs : 5
PASS — raw source contains exactly 5 unique orders.


amount,customer_id,event_time,order_id,order_status
1250.5,501,2026-08-24T09:00:00,1001,COMPLETED
850.0,502,2026-08-24T09:02:00,1002,PENDING
420.75,503,2026-08-24T09:05:00,1003,COMPLETED
990.0,504,2026-08-24T09:07:00,1004,CANCELLED
1750.25,505,2026-08-24T09:10:00,1005,COMPLETED


# 8. What is Auto Loader?

Auto Loader is Databricks' incremental file-ingestion mechanism for cloud storage.

The key source format is:

```python
.format("cloudFiles")
```

Example:

```python
spark.readStream     .format("cloudFiles")
```

Then we tell Auto Loader what the underlying file format is:

```python
.option("cloudFiles.format", "json")
```

So:

```text
cloudFiles
    ↓
Auto Loader
    ↓
JSON files
    ↓
Streaming DataFrame
```

# 9. Build the Auto Loader stream

Now we create a streaming DataFrame.

Notice that this cell **does not execute the complete ingestion yet**.

It defines the streaming source:

```text
Volume files
     ↓
Auto Loader
     ↓
streaming_df
```

In [0]:
orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .load(SOURCE_PATH)
)

print("PASS — Auto Loader streaming DataFrame created.")


PASS — Auto Loader streaming DataFrame created.


# 10. Add basic transformations

We can transform a streaming DataFrame using the normal Spark DataFrame API.

For example:

- convert `event_time`
- add ingestion timestamp
- retain source file information

Unity Catalog does not support the old `input_file_name()` approach used in some older examples.

Instead, use Auto Loader's `_metadata` column.

```text
_metadata.file_path
```

This is the correct approach for this Unity Catalog example.

In [0]:
bronze_df = (
    orders_stream
        .withColumn(
            "event_time",
            F.to_timestamp("event_time")
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

print("PASS — streaming transformations defined.")


PASS — streaming transformations defined.


# 11. First ingestion — `availableNow`

We will use:

```python
.trigger(availableNow=True)
```

This is easier to understand than maintaining a permanently running stream.

Conceptually:

```text
Start stream
     ↓
Process currently available files
     ↓
Finish
```

For this first run:

```text
3 files
 ↓
5 orders
 ↓
Delta Bronze
```

In [0]:
query = (
    bronze_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print("First incremental ingestion completed.")


First incremental ingestion completed.


# 12. Validate the first ingestion

The Bronze table should contain:

```text
5 rows
5 unique order IDs
```

This is our first actual proof that:

```text
Auto Loader
    ↓
Structured Streaming
    ↓
Delta
```

worked for the initial source.

In [0]:
bronze_result = spark.table(BRONZE_TABLE)

row_count = bronze_result.count()
distinct_ids = bronze_result.select("order_id").distinct().count()

print("Bronze rows       :", row_count)
print("Distinct order IDs:", distinct_ids)

assert row_count == 5
assert distinct_ids == 5

print("PASS — initial Auto Loader ingestion produced 5 unique orders.")

display(
    bronze_result
        .select(
            "order_id",
            "customer_id",
            "amount",
            "order_status",
            "event_time",
            "_ingested_at",
            "_source_file"
        )
        .orderBy("order_id")
)


Bronze rows       : 5
Distinct order IDs: 5
PASS — initial Auto Loader ingestion produced 5 unique orders.


order_id,customer_id,amount,order_status,event_time,_ingested_at,_source_file
1001,501,1250.5,COMPLETED,2026-08-24T09:00:00.000Z,2026-08-24T11:14:17.361Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_001.json
1002,502,850.0,PENDING,2026-08-24T09:02:00.000Z,2026-08-24T11:14:17.361Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_001.json
1003,503,420.75,COMPLETED,2026-08-24T09:05:00.000Z,2026-08-24T11:14:17.361Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_002.json
1004,504,990.0,CANCELLED,2026-08-24T09:07:00.000Z,2026-08-24T11:14:17.361Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_002.json
1005,505,1750.25,COMPLETED,2026-08-24T09:10:00.000Z,2026-08-24T11:14:17.361Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_003.json


# 13. What is Structured Streaming?

Structured Streaming is Spark's incremental processing engine.

The pattern is:

```text
readStream
    ↓
transform
    ↓
writeStream
```

In our notebook:

```text
spark.readStream
       ↓
cloudFiles
       ↓
transformations
       ↓
writeStream
       ↓
Delta
```

This is the core Structured Streaming pattern you need for this blog.

# 14. Add a new source file

Now comes the most important practical demonstration.

The original stream processed:

```text
orders_001.json
orders_002.json
orders_003.json
```

Now we add:

```text
orders_004.json
```

with two new orders.

The existing checkpoint will be reused.

The expected result is:

```text
Previous data = 5
New data      = 2
Final table   = 7
```

In [0]:
new_orders = [
    {
        "order_id": 1006,
        "customer_id": 506,
        "amount": 640.00,
        "order_status": "COMPLETED",
        "event_time": "2026-08-24T09:15:00"
    },
    {
        "order_id": 1007,
        "customer_id": 507,
        "amount": 1120.00,
        "order_status": "PENDING",
        "event_time": "2026-08-24T09:18:00"
    }
]

dbutils.fs.put(
    f"{SOURCE_PATH}/orders_004.json",
    "\n".join(json.dumps(record) for record in new_orders),
    True
)

print("New file added: orders_004.json")

Wrote 242 bytes.
New file added: orders_004.json


# 15. Re-create the stream using the SAME checkpoint

This is the key lesson.

We create a new streaming query but point it at:

```text
SAME source
SAME schema location
SAME checkpoint
```

The checkpoint allows the stream to retain knowledge of its previous progress.

This is not the same as creating a brand-new checkpoint.

In [0]:
orders_stream_2 = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .load(SOURCE_PATH)
)

bronze_df_2 = (
    orders_stream_2
        .withColumn(
            "event_time",
            F.to_timestamp("event_time")
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

query_2 = (
    bronze_df_2.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query_2.awaitTermination()

print("Second incremental ingestion completed.")


Second incremental ingestion completed.


# 16. Validate incremental ingestion

Now we expect:

```text
Initial rows = 5
New rows     = 2
Final rows   = 7
```

More importantly:

```text
Distinct order IDs = 7
```

This demonstrates the central Auto Loader concept:

> **New files can be incrementally discovered and processed without treating the old files as new data again.**

In [0]:
final_bronze = spark.table(BRONZE_TABLE)

final_rows = final_bronze.count()
final_ids = final_bronze.select("order_id").distinct().count()

print("Final Bronze rows       :", final_rows)
print("Final distinct order IDs:", final_ids)

assert final_rows == 7
assert final_ids == 7

print("PASS — incremental ingestion added exactly 2 new orders.")
display(
    final_bronze
        .select(
            "order_id",
            "customer_id",
            "amount",
            "order_status",
            "event_time",
            "_source_file"
        )
        .orderBy("order_id")
)


Final Bronze rows       : 7
Final distinct order IDs: 7
PASS — incremental ingestion added exactly 2 new orders.


order_id,customer_id,amount,order_status,event_time,_source_file
1001,501,1250.5,COMPLETED,2026-08-24T09:00:00.000Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_001.json
1002,502,850.0,PENDING,2026-08-24T09:02:00.000Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_001.json
1003,503,420.75,COMPLETED,2026-08-24T09:05:00.000Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_002.json
1004,504,990.0,CANCELLED,2026-08-24T09:07:00.000Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_002.json
1005,505,1750.25,COMPLETED,2026-08-24T09:10:00.000Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_003.json
1006,506,640.0,COMPLETED,2026-08-24T09:15:00.000Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_004.json
1007,507,1120.0,PENDING,2026-08-24T09:18:00.000Z,/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/blog12_runs/20260824_111220/source/orders_004.json


# 17. What did the checkpoint actually do?

The important mental model is:

```text
First run
   ↓
process available files
   ↓
checkpoint records streaming progress

Second run
   ↓
same source
same checkpoint
   ↓
new files are processed incrementally
```

Do not think:

> "Checkpoint stores the data."

Instead think:

> **Checkpoint stores streaming progress and state needed by the query.**

The actual records are stored in the Delta table.

# 18. Schema inference

Auto Loader can infer the schema of source files.

Our source contains:

```text
order_id
customer_id
amount
order_status
event_time
```

Auto Loader needs to determine appropriate Spark data types.

Schema inference is convenient, especially during ingestion development.

In production, important source contracts should be considered deliberately rather than relying blindly on inferred types.

# 19. Schema hints

Schema hints allow you to tell Auto Loader how selected fields should be interpreted.

Example:

```python
.option(
    "cloudFiles.schemaHints",
    "order_id long, customer_id long, amount double"
)
```

This is useful when the source system's representation needs a known interpretation.

Remember:

```text
Schema hint
    ↓
How a field should be interpreted

Schema evolution
    ↓
How structural changes are handled
```

In [0]:
# Example only — this creates a new stream definition.
# It is not executed against the existing production-style query.

orders_stream_with_hints = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{RUN_ROOT}/schema_hints")
        .option(
            "cloudFiles.schemaHints",
            "order_id long, customer_id long, amount double"
        )
        .load(SOURCE_PATH)
)

print("PASS — schema hints were defined.")


PASS — schema hints were defined.


# 20. Schema evolution

Suppose a future file contains:

```text
order_id
customer_id
amount
order_status
event_time
payment_method
```

A new column has appeared.

Auto Loader supports configurable schema evolution behavior.

Common concepts include:

```text
addNewColumns
rescue
failOnNewColumns
```

For this notebook, we will **not deliberately break the stream** with schema changes.

That behavior was already studied in Blog 7.

Here we only need to understand:

> **Auto Loader has schema-evolution policies that determine what happens when source structure changes.**

# 21. Event time

Our records contain:

```text
event_time
```

This represents when the business event occurred.

For example:

```text
Order created
09:10
```

But Spark might process it later:

```text
Processing time
09:15
```

Therefore:

```text
Event time
    ≠
Processing time
```

This distinction becomes important when working with late-arriving data and time-based streaming operations.

In [0]:
# Inspect event time

display(
    final_bronze
        .select(
            "order_id",
            "event_time",
            "_ingested_at"
        )
        .orderBy("event_time")
)


order_id,event_time,_ingested_at
1001,2026-08-24T09:00:00.000Z,2026-08-24T11:14:17.361Z
1002,2026-08-24T09:02:00.000Z,2026-08-24T11:14:17.361Z
1003,2026-08-24T09:05:00.000Z,2026-08-24T11:14:17.361Z
1004,2026-08-24T09:07:00.000Z,2026-08-24T11:14:17.361Z
1005,2026-08-24T09:10:00.000Z,2026-08-24T11:14:17.361Z
1006,2026-08-24T09:15:00.000Z,2026-08-24T11:15:15.538Z
1007,2026-08-24T09:18:00.000Z,2026-08-24T11:15:15.538Z


# 22. Watermarks — concept only

A watermark is used in supported stateful Structured Streaming operations to manage event-time state.

Example syntax:

```python
.withWatermark("event_time", "10 minutes")
```

Conceptually:

```text
Events arrive
     ↓
Event-time progress advances
     ↓
Watermark advances
     ↓
Old state can eventually be considered complete
```

### Important

A watermark is **not simply a filter saying "delete anything older than ten minutes."**

Its main purpose is to help bound state for event-time processing and deal with late data.

For Blog 12, the concept is enough.

# 23. Auto Loader vs ordinary file streaming

You can encounter both:

```python
.format("json")
```

and:

```python
.format("cloudFiles")
```

The important distinction is that Auto Loader is specifically designed for scalable incremental file discovery and ingestion.

For a production cloud-file ingestion workload:

```text
Cloud Storage
      ↓
Auto Loader
      ↓
Structured Streaming
```

is the important pattern to recognize.

# 24. Auto Loader vs Kafka

These technologies solve different source problems.

### Auto Loader

```text
Cloud files
    ↓
Auto Loader
```

### Kafka

```text
Event/message stream
    ↓
Structured Streaming
```

So:

> **Auto Loader is primarily a cloud-file ingestion mechanism. Kafka is an event-streaming platform.**

You do not need Kafka implementation details to complete this blog.

# 25. Auto Loader + Jobs

Blog 11 and Blog 12 now connect.

Blog 11:

```text
Job
 ↓
orchestrates workload
```

Blog 12:

```text
Auto Loader
 ↓
incremental ingestion
```

Together:

```text
                    JOB
                     │
                     ▼
              Auto Loader Task
                     │
                     ▼
                Bronze Delta
                     │
                     ▼
                  Silver
                     │
                     ▼
                   Gold
```

A scheduled Job can therefore run an incremental ingestion workload.

# 26. Auto Loader + Unity Catalog

Blog 10 taught:

```text
Catalog
   ↓
Schema
   ↓
Volume
```

We are using:

```text
workspace
   ↓
blog10_autoloader_streaming
   ↓
blog10_volume
```

Therefore:

```text
/Volumes/workspace/blog10_autoloader_streaming/blog10_volume/
```

is a Unity Catalog Volume path.

The data target is also a fully qualified UC table:

```text
workspace.blog10_autoloader_streaming.<table>
```

### Mental model

```text
Unity Catalog
     ↓
governed storage + tables

Auto Loader
     ↓
ingestion

Structured Streaming
     ↓
incremental processing

Delta
     ↓
transactional storage
```

# 27. Production architecture

Putting the concepts together:

```text
                    CLOUD STORAGE
                          │
                          ▼
                     AUTO LOADER
                          │
                          ▼
                STRUCTURED STREAMING
                          │
                          ▼
                  UNITY CATALOG
                          │
                          ▼
                    DELTA BRONZE
                          │
                          ▼
                       SILVER
                          │
                          ▼
                        GOLD
```

And around the pipeline:

```text
Jobs       → orchestration
Unity Cat. → governance
Delta      → transactional storage
AutoLoader → file ingestion
Streaming  → incremental execution
```

# 28. What you should know after Blog 12

You should be comfortable explaining:

### Incremental processing

> Process newly available data rather than repeatedly processing everything.

### Structured Streaming

> Spark's incremental processing engine using the DataFrame/SQL programming model.

### Auto Loader

> Databricks' incremental file-ingestion mechanism for cloud storage.

### `cloudFiles`

> The Auto Loader source format.

### Schema location

> Stores Auto Loader source-schema information.

### Checkpoint

> Stores streaming progress and state.

### `availableNow`

> Processes currently available data and then stops.

### Event time

> The time when the business event occurred.

### Watermark

> A mechanism for managing event-time state in supported stateful operations.

### Schema evolution

> The configured behavior when the source structure changes.

# 29. Practical checklist

Before considering Blog 12 complete, you should be able to say:

- [x] I understand batch vs incremental processing.
- [x] I know what Structured Streaming is.
- [x] I know `readStream` and `writeStream`.
- [x] I know what Auto Loader is.
- [x] I know `cloudFiles`.
- [x] I know why schema locations exist.
- [x] I know why checkpoints exist.
- [x] I used `availableNow`.
- [x] I ingested files into Delta.
- [x] I added a new file after the first run.
- [x] I reused the same checkpoint.
- [x] I validated incremental ingestion.
- [x] I understand schema hints.
- [x] I understand schema evolution conceptually.
- [x] I understand event time.
- [x] I understand the purpose of watermarks.
- [x] I understand Auto Loader vs Kafka.
- [x] I understand how Auto Loader connects to Jobs and Unity Catalog.

# 30. What we deliberately did NOT cover

To keep this blog practical without making it overwhelming, we did **not** build:

- complex streaming joins
- state-store internals
- advanced stateful aggregations
- complicated late-data experiments
- Kafka clusters
- continuous processing
- advanced streaming performance tuning
- complicated failure injection
- advanced schema-failure labs

These are advanced topics.

The objective here is to establish a **solid working understanding of the normal Auto Loader + Structured Streaming workflow**.

# 31. Final mental model

```text
                 CLOUD STORAGE
                      │
                      ▼
                 AUTO LOADER
                      │
                discovers files
                      │
                      ▼
            STRUCTURED STREAMING
                      │
               incremental
                processing
                      │
                      ▼
                DELTA BRONZE
                      │
                      ▼
                   SILVER
                      │
                      ▼
                    GOLD
```

Supporting components:

```text
Schema Location
      ↓
Auto Loader schema information

Checkpoint
      ↓
Streaming progress/state

Unity Catalog
      ↓
Governance

Jobs
      ↓
Orchestration
```

### One sentence to remember

> **Auto Loader discovers newly arriving cloud files, Structured Streaming processes them incrementally, checkpoints maintain streaming progress, and Delta provides the transactional destination.**

This is the practical core you need before moving into the **final end-to-end Databricks project**.